Step 1 — Initialize Snowpark Session

This:
connects Python to Snowflake

enables DataFrame operations

allows querying warehouse tables directly

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col,
    sum as sf_sum,
    count,
    avg,
    round as sf_round,
    when
)

# Get active Snowflake session
session = get_active_session()

print("Snowpark Session Initialized Successfully")

Step 2 — Load FACT_TRANSACTIONS Data

querying Snowflake tables via Python

using Snowpark DataFrames

performing distributed processing inside Snowflake

In [ ]:
fact_df = session.table("SYNTHETIC_FINANCIAL_DB.GOLD_LAYER.FACT_TRANSACTIONS")

fact_df.show(5)

Step 3 — Create Customer Fraud Metrics

For each customer:

total activity

spending behavior

fraud involvement

transaction metrics

In [ ]:
# customer level fraud analytics

customer_risk_df = (
    fact_df
    .group_by(col("CUSTOMER_KEY"))
    .agg(
        count("*").alias("TOTAL_TRANSACTIONS"),

        sf_round(
            sf_sum(col("TRANSACTION_AMOUNT")), 2
        ).alias("TOTAL_TRANSACTION_AMOUNT"),

        sf_round(
            avg(col("TRANSACTION_AMOUNT")), 2
        ).alias("AVG_TRANSACTION_AMOUNT"),

        sf_sum(col("IS_FRAUD"))
            .alias("FRAUD_TRANSACTIONS")
    )
)

customer_risk_df.show(10)

Step 4 — Create Fraud Risk Score

This demonstrates:

rule-based ML-style scoring

fraud intelligence engineering

Python feature processing

Snowpark transformations

In [ ]:
# Create fraud risk score

customer_risk_df = (
    customer_risk_df
    .with_column(
        "FRAUD_RATE_PERCENT",
        sf_round(
            (col("FRAUD_TRANSACTIONS") * 100.0)
            / col("TOTAL_TRANSACTIONS"),
            4
        )
    )
    .with_column(
        "RISK_SCORE",

        when(col("FRAUD_TRANSACTIONS") > 0, 90)

        .when(col("AVG_TRANSACTION_AMOUNT") >= 500000, 75)

        .when(col("AVG_TRANSACTION_AMOUNT") >= 100000, 50)

        .otherwise(20)
    )
    .with_column(
        "RISK_CATEGORY",

        when(col("RISK_SCORE") >= 80, "HIGH_RISK")

        .when(col("RISK_SCORE") >= 50, "MEDIUM_RISK")

        .otherwise("LOW_RISK")
    )
)

# Preview scored customers
customer_risk_df.show(20)

Step 5 — Save Results Back to Snowflake

processing data in Python

inside Snowflake compute

writing results back to warehouse

In [ ]:
# Save fraud risk scores table

customer_risk_df.write.mode("overwrite").save_as_table(
    "SYNTHETIC_FINANCIAL_DB.GOLD_LAYER.CUSTOMER_FRAUD_RISK_SCORES"
)

print("Fraud Risk Scores Table Created Successfully")